# 📊 Notebook 01: Data Preparation
## Diabetic Retinopathy Grading System - Week 1

**Objectives:**
1. ✅ Verify dataset structure and integrity
2. ✅ Analyze class distribution
3. ✅ Create sealed stratified splits (Train/Val/Test)
4. ✅ Perform pHash duplicate check
5. ✅ Generate data manifest

**Expected Output:**
- `splits/train.csv` (2,592 samples)
- `splits/val.csv` (459 samples)
- `splits/test.csv` (611 samples)
- `results/data_manifest.json`

---

In [ ]:
%pip install imagehash

## 1. Setup & Imports

In [ ]:
# Core imports
import os
import sys
import json
import hashlib
from pathlib import Path
from datetime import datetime
from collections import Counter

# Data processing
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Image processing
from PIL import Image
import imagehash

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress bar
from tqdm.notebook import tqdm

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")

In [ ]:
# Define paths - Data is now inside the project folder
from pathlib import Path
import os

# Project root (absolute path)
PROJECT_ROOT = Path("/Users/shivasaivemula/ALP Project/deep-retina-grade")

# Data directories - datasets are now inside the project folder
DATA_ROOT = PROJECT_ROOT / "aptos2019-blindness-detection"
MESSIDOR_ROOT = PROJECT_ROOT / "archive"

# Project directories
SPLITS_DIR = PROJECT_ROOT / "splits"
RESULTS_DIR = PROJECT_ROOT / "results"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

# Data paths
TRAIN_CSV = DATA_ROOT / "train.csv"
TRAIN_IMAGES_DIR = DATA_ROOT / "train_images"
TEST_IMAGES_DIR = DATA_ROOT / "test_images"

# Verify paths exist
print("📁 Path Verification:")
print(f"  Project Root: {PROJECT_ROOT} - {'✅' if PROJECT_ROOT.exists() else '❌'}")
print(f"  Data Root: {DATA_ROOT} - {'✅' if DATA_ROOT.exists() else '❌'}")
print(f"  Train CSV: {TRAIN_CSV} - {'✅' if TRAIN_CSV.exists() else '❌'}")
print(f"  Train Images: {TRAIN_IMAGES_DIR} - {'✅' if TRAIN_IMAGES_DIR.exists() else '❌'}")
print(f"  Messidor Data: {MESSIDOR_ROOT} - {'✅' if MESSIDOR_ROOT.exists() else '❌'}")

# List what's in DATA_ROOT to confirm
if DATA_ROOT.exists():
    print(f"\n📂 Contents of DATA_ROOT:")
    for item in DATA_ROOT.iterdir():
        print(f"  {'📁' if item.is_dir() else '📄'} {item.name}")

## 2. Load and Verify Dataset

In [ ]:
# Load training labels
df_train = pd.read_csv(TRAIN_CSV)

print("📊 Dataset Overview:")
print(f"  Total samples: {len(df_train):,}")
print(f"  Columns: {list(df_train.columns)}")
print(f"\n🔍 First 10 samples:")
df_train.head(10)

In [ ]:
# Verify all images exist
print("🔍 Verifying image files...")

missing_images = []
valid_images = []

for idx, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Checking images"):
    img_path = TRAIN_IMAGES_DIR / f"{row['id_code']}.png"
    if img_path.exists():
        valid_images.append(row['id_code'])
    else:
        missing_images.append(row['id_code'])

print(f"\n📊 Image Verification Results:")
print(f"  ✅ Valid images: {len(valid_images):,}")
print(f"  ❌ Missing images: {len(missing_images)}")

if missing_images:
    print(f"  Missing: {missing_images[:5]}...")

In [ ]:
# Filter to only valid images
df_train = df_train[df_train['id_code'].isin(valid_images)].reset_index(drop=True)
print(f"✅ Working with {len(df_train):,} valid samples")

## 3. Class Distribution Analysis

**DR Grades:**
- 0: No DR (Healthy)
- 1: Mild NPDR
- 2: Moderate NPDR
- 3: Severe NPDR
- 4: Proliferative DR (Most severe)

In [ ]:
# Analyze class distribution
class_names = {
    0: "No DR",
    1: "Mild NPDR",
    2: "Moderate NPDR",
    3: "Severe NPDR",
    4: "Proliferative DR"
}

class_distribution = df_train['diagnosis'].value_counts().sort_index()

print("📊 Class Distribution:")
print("=" * 50)
for grade, count in class_distribution.items():
    percentage = count / len(df_train) * 100
    print(f"  Grade {grade} ({class_names[grade]}): {count:,} ({percentage:.1f}%)")
print("=" * 50)
print(f"  Total: {len(df_train):,}")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#9b59b6']
bars = axes[0].bar(range(5), class_distribution.values, color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_xlabel('DR Grade', fontsize=12)
axes[0].set_ylabel('Number of Samples', fontsize=12)
axes[0].set_title('Class Distribution - APTOS 2019 Dataset', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(5))
axes[0].set_xticklabels([f"Grade {i}\n{class_names[i]}" for i in range(5)], fontsize=10)

# Add count labels on bars
for bar, count in zip(bars, class_distribution.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
                 f'{count:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Pie chart
explode = (0.02, 0.02, 0.02, 0.05, 0.05)  # Highlight severe cases
wedges, texts, autotexts = axes[1].pie(
    class_distribution.values, 
    labels=[f"Grade {i}" for i in range(5)],
    autopct='%1.1f%%',
    colors=colors,
    explode=explode,
    startangle=90,
    wedgeprops={'edgecolor': 'black', 'linewidth': 1}
)
axes[1].set_title('Percentage Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'class_distribution.png'}")

In [ ]:
# Calculate class imbalance metrics
majority_class = class_distribution.max()
minority_class = class_distribution.min()
imbalance_ratio = majority_class / minority_class

print("⚠️ Class Imbalance Analysis:")
print(f"  Majority class (Grade 0): {majority_class:,} samples")
print(f"  Minority class (Grade {class_distribution.idxmin()}): {minority_class:,} samples")
print(f"  Imbalance ratio: {imbalance_ratio:.1f}:1")
print(f"\n  → This confirms the ~70% Grade 0 issue mentioned in the master plan!")
print(f"  → Will use stratified splits + QWK loss to address this.")

## 4. pHash Duplicate Detection

Ensure no duplicate images exist that could cause data leakage between splits.

In [ ]:
# Calculate perceptual hashes for duplicate detection
print("🔍 Calculating perceptual hashes for duplicate detection...")
print("   (This may take a few minutes)")

phash_dict = {}
hash_to_images = {}

for idx, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Computing pHash"):
    img_path = TRAIN_IMAGES_DIR / f"{row['id_code']}.png"
    try:
        img = Image.open(img_path)
        phash = str(imagehash.phash(img))
        phash_dict[row['id_code']] = phash
        
        if phash not in hash_to_images:
            hash_to_images[phash] = []
        hash_to_images[phash].append(row['id_code'])
    except Exception as e:
        print(f"  Error processing {row['id_code']}: {e}")

print(f"\n✅ Computed hashes for {len(phash_dict):,} images")

In [ ]:
# Find duplicates (same hash = potential duplicate)
duplicates = {h: imgs for h, imgs in hash_to_images.items() if len(imgs) > 1}

print("📊 Duplicate Analysis Results:")
print(f"  Unique hashes: {len(hash_to_images):,}")
print(f"  Potential duplicate groups: {len(duplicates)}")

if duplicates:
    print(f"\n⚠️ Found {len(duplicates)} potential duplicate groups:")
    for i, (h, imgs) in enumerate(list(duplicates.items())[:5]):
        print(f"  Group {i+1}: {imgs}")
    
    # Total duplicate images
    total_duplicates = sum(len(imgs) - 1 for imgs in duplicates.values())
    print(f"\n  Total duplicate images to consider: {total_duplicates}")
else:
    print("\n✅ No exact duplicates found! Dataset is clean.")

In [ ]:
# Add pHash to dataframe for tracking
df_train['phash'] = df_train['id_code'].map(phash_dict)
df_train.head()

## 5. Create Stratified Splits

**Target Split Sizes (from Master Plan):**
- Train: ~71% (2,592 samples)
- Validation: ~12.5% (459 samples)
- Test: ~16.5% (611 samples) - **SEALED for final evaluation**

In [ ]:
# Set random seed for reproducibility
RANDOM_SEED = 42

# First split: Separate test set (16.5%)
df_trainval, df_test = train_test_split(
    df_train,
    test_size=0.165,
    stratify=df_train['diagnosis'],
    random_state=RANDOM_SEED
)

# Second split: Separate train and validation from trainval (val ~15% of trainval = 12.5% of total)
df_train_final, df_val = train_test_split(
    df_trainval,
    test_size=0.15,
    stratify=df_trainval['diagnosis'],
    random_state=RANDOM_SEED
)

print("📊 Split Sizes:")
print(f"  Train: {len(df_train_final):,} ({len(df_train_final)/len(df_train)*100:.1f}%)")
print(f"  Val:   {len(df_val):,} ({len(df_val)/len(df_train)*100:.1f}%)")
print(f"  Test:  {len(df_test):,} ({len(df_test)/len(df_train)*100:.1f}%)")
print(f"  Total: {len(df_train_final) + len(df_val) + len(df_test):,}")

In [ ]:
# Verify stratification - class distribution should be similar across splits
def get_distribution(df):
    dist = df['diagnosis'].value_counts().sort_index()
    return (dist / len(df) * 100).round(1)

print("📊 Class Distribution Verification (percentages):")
print("=" * 60)

dist_df = pd.DataFrame({
    'Grade': [f"{i} - {class_names[i]}" for i in range(5)],
    'Train': get_distribution(df_train_final).values,
    'Val': get_distribution(df_val).values,
    'Test': get_distribution(df_test).values,
    'Original': get_distribution(df_train).values
})

print(dist_df.to_string(index=False))
print("=" * 60)
print("✅ Stratification successful - distributions are nearly identical!")

In [ ]:
# Visualize split distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

splits = [
    (df_train_final, 'Train', '#3498db'),
    (df_val, 'Validation', '#2ecc71'),
    (df_test, 'Test (SEALED)', '#e74c3c')
]

for ax, (df, name, color) in zip(axes, splits):
    counts = df['diagnosis'].value_counts().sort_index()
    ax.bar(range(5), counts.values, color=color, edgecolor='black', alpha=0.8)
    ax.set_title(f'{name} Set (n={len(df):,})', fontsize=12, fontweight='bold')
    ax.set_xlabel('DR Grade')
    ax.set_ylabel('Count')
    ax.set_xticks(range(5))
    
    # Add percentage labels
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, f'{v/len(df)*100:.1f}%', ha='center', fontsize=9)

plt.suptitle('Stratified Split Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'split_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'split_distributions.png'}")

In [ ]:
# Verify no data leakage between splits
train_ids = set(df_train_final['id_code'])
val_ids = set(df_val['id_code'])
test_ids = set(df_test['id_code'])

train_val_overlap = train_ids.intersection(val_ids)
train_test_overlap = train_ids.intersection(test_ids)
val_test_overlap = val_ids.intersection(test_ids)

print("🔒 Data Leakage Check:")
print(f"  Train ∩ Val overlap: {len(train_val_overlap)} {'✅' if len(train_val_overlap) == 0 else '❌'}")
print(f"  Train ∩ Test overlap: {len(train_test_overlap)} {'✅' if len(train_test_overlap) == 0 else '❌'}")
print(f"  Val ∩ Test overlap: {len(val_test_overlap)} {'✅' if len(val_test_overlap) == 0 else '❌'}")

if len(train_val_overlap) == 0 and len(train_test_overlap) == 0 and len(val_test_overlap) == 0:
    print("\n✅ No data leakage detected! Splits are clean.")

## 6. Save Splits & Generate Manifest

In [ ]:
# Save splits to CSV files
df_train_final[['id_code', 'diagnosis']].to_csv(SPLITS_DIR / 'train.csv', index=False)
df_val[['id_code', 'diagnosis']].to_csv(SPLITS_DIR / 'val.csv', index=False)
df_test[['id_code', 'diagnosis']].to_csv(SPLITS_DIR / 'test.csv', index=False)

print("💾 Saved split files:")
print(f"  ✅ {SPLITS_DIR / 'train.csv'} ({len(df_train_final):,} samples)")
print(f"  ✅ {SPLITS_DIR / 'val.csv'} ({len(df_val):,} samples)")
print(f"  ✅ {SPLITS_DIR / 'test.csv'} ({len(df_test):,} samples)")

In [ ]:
# Generate data manifest
def calculate_file_hash(filepath):
    """Calculate MD5 hash of a file."""
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

manifest = {
    "project": "Deep Retina Grade - Diabetic Retinopathy Grading System",
    "created_at": datetime.now().isoformat(),
    "random_seed": RANDOM_SEED,
    "dataset": {
        "name": "APTOS 2019 Blindness Detection",
        "source": "Kaggle",
        "total_images": len(df_train),
        "image_format": "PNG",
        "class_distribution": {
            class_names[k]: int(v) for k, v in class_distribution.items()
        }
    },
    "splits": {
        "train": {
            "count": len(df_train_final),
            "percentage": round(len(df_train_final) / len(df_train) * 100, 1),
            "file": "splits/train.csv",
            "checksum": calculate_file_hash(SPLITS_DIR / 'train.csv')
        },
        "validation": {
            "count": len(df_val),
            "percentage": round(len(df_val) / len(df_train) * 100, 1),
            "file": "splits/val.csv",
            "checksum": calculate_file_hash(SPLITS_DIR / 'val.csv')
        },
        "test": {
            "count": len(df_test),
            "percentage": round(len(df_test) / len(df_train) * 100, 1),
            "file": "splits/test.csv",
            "checksum": calculate_file_hash(SPLITS_DIR / 'test.csv'),
            "note": "SEALED - Do not use until final evaluation!"
        }
    },
    "duplicate_check": {
        "method": "pHash (Perceptual Hash)",
        "unique_hashes": len(hash_to_images),
        "potential_duplicate_groups": len(duplicates),
        "status": "CLEAN" if len(duplicates) == 0 else f"{len(duplicates)} groups found"
    },
    "class_imbalance": {
        "majority_class": "Grade 0 (No DR)",
        "minority_class": f"Grade {class_distribution.idxmin()} ({class_names[class_distribution.idxmin()]})",
        "imbalance_ratio": round(imbalance_ratio, 1),
        "mitigation": ["Stratified splits", "Quadratic Weighted Kappa loss", "Class weighting"]
    }
}

# Save manifest
with open(RESULTS_DIR / 'data_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print("📋 Data Manifest Generated:")
print(json.dumps(manifest, indent=2))

## 7. Sample Images Visualization

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for grade in range(5):
    # Get 2 samples from each grade
    grade_samples = df_train_final[df_train_final['diagnosis'] == grade].sample(2, random_state=42)
    
    for row_idx, (_, sample) in enumerate(grade_samples.iterrows()):
        img_path = TRAIN_IMAGES_DIR / f"{sample['id_code']}.png"
        img = Image.open(img_path)
        
        ax = axes[row_idx, grade]
        ax.imshow(img)
        ax.set_title(f"Grade {grade}: {class_names[grade]}", fontsize=11, fontweight='bold')
        ax.axis('off')

plt.suptitle('Sample Fundus Images by DR Grade (Before Preprocessing)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'sample_images_raw.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'sample_images_raw.png'}")

## 8. Summary & Next Steps

In [ ]:
print("="*70)
print("📊 DATA PREPARATION COMPLETE - SUMMARY")
print("="*70)
print(f"""
✅ Dataset Verified:
   • Total samples: {len(df_train):,}
   • All images accessible: Yes
   
✅ Class Distribution Analyzed:
   • Grade 0 (No DR): {class_distribution[0]:,} ({class_distribution[0]/len(df_train)*100:.1f}%)
   • Grade 1 (Mild): {class_distribution[1]:,} ({class_distribution[1]/len(df_train)*100:.1f}%)
   • Grade 2 (Moderate): {class_distribution[2]:,} ({class_distribution[2]/len(df_train)*100:.1f}%)
   • Grade 3 (Severe): {class_distribution[3]:,} ({class_distribution[3]/len(df_train)*100:.1f}%)
   • Grade 4 (Proliferative): {class_distribution[4]:,} ({class_distribution[4]/len(df_train)*100:.1f}%)
   • Imbalance Ratio: {imbalance_ratio:.1f}:1

✅ Stratified Splits Created:
   • Train: {len(df_train_final):,} samples → splits/train.csv
   • Validation: {len(df_val):,} samples → splits/val.csv
   • Test: {len(df_test):,} samples → splits/test.csv (SEALED)

✅ Duplicate Check (pHash):
   • Status: {'CLEAN - No duplicates' if len(duplicates) == 0 else f'{len(duplicates)} potential duplicate groups'}

✅ Data Leakage Check:
   • No overlap between splits: Confirmed

📁 Artifacts Generated:
   • {ARTIFACTS_DIR / 'class_distribution.png'}
   • {ARTIFACTS_DIR / 'split_distributions.png'}
   • {ARTIFACTS_DIR / 'sample_images_raw.png'}
   • {RESULTS_DIR / 'data_manifest.json'}
""")
print("="*70)
print("\n🚀 NEXT STEPS (Notebook 02):")
print("   1. Implement RetinaPreprocessor (Ben Graham + CLAHE)")
print("   2. Create preprocessing visualization grid")
print("   3. Test on young/old/dark-skinned samples")
print("="*70)